# Vector RAG: Grounding LLMs with Semantic Search

**Objective:** 
In this lesson, we will explore a Retrieval-Augmented Generation (RAG) system built using a **Vector Database** (FAISS). This provides a contrast to the Graph Database approach we explored previously, allowing you to understand how Semantic Search processes text into vector embeddings to retrieve factual context and ground LLM responses.

---

In this notebook, we are going to fix **The Knowledge Gap** by teaching our AI to read *your* private technical manuals before it answers.

### Concept: The Knowledge Gap and RAG

Large Language Models are trained on a massive snapshot of the public internet. If a technical manual was written after its training cutoff date, or if it's a private company document, the LLM simply hasn't read it. If you ask it a question about that document, it will either say "I don't know" or worse, it will hallucinate (make something up).

**The Solution: Retrieval-Augmented Generation (RAG)**
Instead of retraining the entire model (which is wildly expensive), we give the LLM an open-book test.

Here is the architecture we will build today:
1. **Ingestion (PDF):** We load your technical manual.
2. **Chunking:** We split the manual into smaller, bite-sized paragraphs.
3. **Embeddings:** We convert these text chunks into numerical vectors (lists of numbers) that capture their meaning.
4. **Vector Database (FAISS):** We store these vectors in a specialized database.
5. **Retrieval:** When a user asks a question, we convert the question into a vector, search the database for the top 3 most similar chunks, and retrieve them.
6. **Generation (LLM):** We feed those specific chunks to Llama 3.2 (running locally via Ollama) along with the question and say, *"Based ONLY on these chunks, answer the question."*

### Step 1: Setting up our RAG Tools

To build our pipeline, we need a few tools. We will use **LangChain** to orchestrate, and **Ollama** to run models locally.

| Library | Purpose |
| :--- | :--- |
| **`langchain` & `langchain-ollama`** | The orchestration framework and Ollama integration. |
| **`pypdf`** | A tool to extract text from PDF files. |
| **`faiss`** | A high-performance Vector Database created by Meta to store and search our embeddings locally. |

In [ ]:
# Run this cell to install the necessary libraries into your Anaconda environment
%pip install -q langchain langchain-community langchain-ollama pypdf faiss-cpu langchain-text-splitters

### Step 2: Ensure Local Ollama is Running & Pull Models

Since we are running this locally in Anaconda, ensure that your **Ollama desktop application is running in the background** before executing this cell.

We will download the models we need:
- **llama3.2**: For generating text.
- **nomic-embed-text**: For generating vector embeddings.

In [ ]:
import subprocess

# Pull the necessary models (this might take a minute if you haven't downloaded them yet)
print("Pulling llama3.2 model...")
subprocess.run(["ollama", "pull", "llama3.2"])

print("Pulling nomic-embed-text model...")
subprocess.run(["ollama", "pull", "nomic-embed-text"])

print("\nLocal Ollama models are ready!")

### Step 3: Loading the PDF and Splitting into Chunks

We can't feed a 100-page manual into an LLM all at once; it will forget things and run out of memory. We need to cut the document into overlapping chunks.

*Before running the next cell, make sure you have the PDF file named `sample_manual.pdf` in the same directory as this notebook!*

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load the PDF
loader = PyPDFLoader("sample_manual.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages from the PDF.")

# 2. Split the text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(documents)
print(f"Split the document into {len(chunks)} chunks.")

### Step 4: Embeddings and the FAISS Vector Database

Now we translate our text chunks into vectors using Ollama's `nomic-embed-text` model. Then, we load these vectors into **FAISS**, which acts as our search engine.

In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Load the embedding model
print("Loading Ollama embedding model...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 2. Create the Vector Database
print("Creating FAISS Vector Database. This might take a moment...")
vector_db = FAISS.from_documents(chunks, embeddings)

# 3. Create a retriever object that fetches the top 3 most relevant chunks
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector Database is ready!")

### Step 5: Building the "Manual Reader" Prompt

Now we tie it all together. When a user asks a question, we will:
1. Search FAISS for the top 3 chunks.
2. Inject those chunks into a strict prompt.
3. Send the prompt to Llama 3.2 via Ollama.

In [ ]:
from langchain_ollama import ChatOllama

# The user's question
question = "What is the recommended operating temperature for the main sensor?"

# 1. Retrieve the relevant chunks from the database
retrieved_docs = retriever.invoke(question)

# Combine the retrieved text into a single string
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# 2. Format the prompt
prompt = f"""You are an expert technical assistant. Answer the user's question based ONLY on the context provided below.
If the answer is not contained in the context, say "I cannot find the answer in the provided manual." Do not use outside knowledge.

Context:
{context}

Question: {question}
"""

print("Generating answer...\n")

# 3. Generate the response using Ollama
llm = ChatOllama(model="llama3.2", temperature=0.1)
response = llm.invoke(prompt)

# Print the answer
print(f"Question: {question}\n")
print(f"Answer: {response.content}")

## Practice Exercises

**Exercise 1: Querying Troubleshooting Protocols**
Use the cell below to ask the AI: *"What should I do if I get Error Code 502-V?"* Verify that the RAG pipeline successfully extracts the correct maintenance steps from the manual.

**Exercise 2: Context Guardrail Testing**
Test the "Knowledge Gap" explicitly by asking the AI a question that is **not** covered in the `sample_manual.pdf` (e.g., *"How do I replace the robotic arm's hydraulic fluid?"*). Ensure that the AI successfully relies on its prompt guardrail and refuses to hallucinate an answer.

**Exercise 3: Vector DB vs. Graph DB Reflection**
Think back to the Neo4j Graph Database we built previously. How does this Vector Database (FAISS) approach compare? 
- How does splitting a document into overlap chunks differ from explicitly extracting relationships between entities?
- In what scenarios would you prefer Vector RAG (Semantic Search) over GraphRAG (Knowledge Engineering)?

In [ ]:
# Write your code for Exercise 1 & 2 here:
# Hint: Redefine the 'question' variable, retrieve documents, structure the prompt, and invoke the LLM.



In [ ]:
# Create a reusable function to query the pipeline
def ask_manual(query):
    retrieved_docs = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""You are an expert technical assistant. Answer the user's question based ONLY on the context provided below.
    If the answer is not contained in the context, say "I cannot find the answer in the provided manual." Do not use outside knowledge.

    Context:
    {context}

    Question: {query}
    """
    response = llm.invoke(prompt)
    print(f"Question: {query}\nAnswer: {response.content}\n" + "-"*50)

# Exercise 1: Querying Troubleshooting Protocols
ask_manual("What should I do if I get Error Code 502-V?")

# Exercise 2: Context Guardrail Testing
ask_manual("How do I replace the robotic arm's hydraulic fluid?")

### Exercise 3: Vector DB vs. Graph DB Reflection

* **Chunking vs. Explicit Extraction:** Splitting documents into overlapping chunks cuts text mathematically based on character limits, grabbing raw paragraphs without understanding the underlying concepts. Graph extraction explicitly defines logic (e.g., `(Drug)-[:CAUSES]->(Adverse Event)`), requiring heavier upfront engineering but guaranteeing the AI only retrieves strict factual relationships free of surrounding document noise.
* **When to Use Vector RAG (Semantic Search):** Ideal for unstructured data like HR handbooks, lengthy policies, customer support logs, or generalized manuals. It excels when users ask broad questions and need surrounding textual context retrieved by conceptual similarity.
* **When to Use GraphRAG (Knowledge Engineering):** Ideal for strict, relational data like medical contraindications, financial compliance, or supply chains. It excels at multi-hop reasoning where precision is critical and statistical guessing could lead to dangerous hallucinations.